# LookTwice — 02 Training (Kaggle)

**Settings:** Accelerator = GPU (T4/P100), Internet = ON (pip + download bobot pretrained ImageNet dari Hugging Face).

**Add data:**
- dataset crop hasil notebook 01 (`looktwice-faces`)
- kode repo (GitHub atau Kaggle Dataset `looktwice-src`)
- *(opsional)* output versi sebelumnya dari notebook ini, untuk melanjutkan run yang belum selesai

**Bagi kerja antar anggota:** tiap orang bisa menjalankan satu model di akun Kaggle masing-masing, mis. `ONLY = ['resnet50']`, lalu gabungkan folder `runs/`.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/garrentanavaro/look-twice.git'
REPO_DATASET = ''
WORK = '/kaggle/working/look-twice'
if not os.path.exists(WORK):
    if REPO_URL: subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, WORK], check=True)
    else: subprocess.run(['cp', '-r', REPO_DATASET, WORK], check=True)
os.chdir(WORK)
!pip install -q -e . timm "albumentations>=2.0"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
INDEX = '/kaggle/input/looktwice-faces/faces/crops_index.csv'
RUNS = '/kaggle/working/runs'
PREV_RUNS = ''
if PREV_RUNS:
    !mkdir -p $RUNS && cp -rn $PREV_RUNS/* $RUNS/
import pandas as pd
idx = pd.read_csv(INDEX); print(idx.groupby(['split', 'label']).size().unstack())

## 0. Sanity run (±2 menit)
Run kecil untuk memastikan pipeline, GPU, dan download bobot berjalan sebelum menghabiskan kuota.

In [ ]:
!python -m looktwice.train --config configs/train/base.yaml --config configs/train/recipe_B.yaml \n    --set data.index=$INDEX --set out_dir=/kaggle/working/sanity --set run_name=sanity \n    --set data.max_train=512 --set data.max_val=256 --set train.epochs=2

## 1. Stage 1: 3 model × 3 recipe
Run yang sudah punya `summary.json` otomatis dilewati.

In [ ]:
ONLY = []
only_args = ' '.join(f'--only {o}' for o in ONLY)
!python -m looktwice.run_experiments --stage stage1 $only_args --set data.index=$INDEX --set out_dir=$RUNS

## Kurva training
Untuk analisis overfitting / underfitting / stabilitas (brief §12).

In [ ]:
import matplotlib.pyplot as plt, glob
for h in sorted(glob.glob(f'{RUNS}/*/history.csv')):
    d = pd.read_csv(h); name = h.split('/')[-2]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.2)); fig.suptitle(name)
    ax[0].plot(d.epoch, d.train_loss, label='train'); ax[0].plot(d.epoch, d.val_loss, label='val'); ax[0].set_title('loss'); ax[0].legend()
    ax[1].plot(d.epoch, d.train_acc, label='train acc'); ax[1].plot(d.epoch, d.val_acc, label='val acc'); ax[1].plot(d.epoch, d.val_auc, label='val AUC'); ax[1].legend(); ax[1].set_title('metric')
    ax[2].plot(d.epoch, d.lr); ax[2].set_yscale('log'); ax[2].set_title('learning rate')
    plt.tight_layout(); plt.show()

In [ ]:
!python -m looktwice.run_experiments --collect --set out_dir=$RUNS
pd.read_csv('results/runs_summary.csv').sort_values('val_auc', ascending=False)

## 2. Stage 2 & 3
Isi `stage2.model/recipe` dan `stage3.model/recipe` di `configs/train/experiments.yaml` berdasarkan tabel di atas (commit perubahan itu ke repo), lalu jalankan:

## Simpan
`best.pt` ±70–100 MB per run. Output `/kaggle/working/runs` + `results/runs_summary.csv` tersimpan saat *Save Version*. Download `results/` + `history.csv` untuk di-commit; bobot cukup yang terpilih saja.

In [ ]:
!rm -rf /kaggle/working/sanity
!cp -r results /kaggle/working/results_table
!du -sh $RUNS